In [0]:
silver_path_movies = "/Volumes/workspace/default/myspace/movies/silver/movies/movies.parquet"
silver_path_ratings = "/Volumes/workspace/default/myspace/movies/silver/ratings/ratings.parquet"

movies_df = (spark.read.format("parquet")
      .option("recursiveFileLookup", "true")
      .load(silver_path_movies)
)

ratings_df = (spark.read.format("parquet")
      .option("recursiveFileLookup", "true")
      .load(silver_path_ratings)
)

In [0]:
movies_df.show(5)

In [0]:
ratings_df.show(5)

In [0]:
# get movie wise average rating, and movies wise count of ratings recieved
# keep only movie data having more than 100 ratings recieved and average rating more than or equal to 3.5
# sort the finalized data in descending order of ratings count

import pyspark.sql.functions as F

mostPopularMoviesDf = (
                        ratings_df
                       .groupBy("movieId")
                       .agg(F.avg("rating").alias("avg_rating"), F.count("userId").alias("total_ratings"))
                       .filter((F.col("total_ratings") >= 100) & (F.col("avg_rating") >=3.5))
                       .sort(F.desc("total_ratings"))
)

In [0]:
mostPopularMoviesDf.show()

In [0]:
mostPopularMoviesDf.explain(extended=True)

In [0]:
gold_popular_movies = "/Volumes/workspace/default/myspace/movies/gold/movies/movies.parquet"

mostPopularMoviesDf.write.parquet(path=gold_popular_movies, mode='overwrite')


=========================================== END ==============================================